# Diabetes Risk Prediction

## Setup

In [ ]:
import os
os.environ["JOBLIB_MULTIPROCESSING"] = "0"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from sklearn import set_config
set_config(transform_output="default")

import numpy as np, pandas as pd, sklearn
print(f"numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__}")

## Load data

In [ ]:
import pandas as pd
from pathlib import Path

CSV_PATH = "diabetes_sample_5000.csv"
TARGET_COL = "diabetes_label"

if not Path(CSV_PATH).exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
df['gender'] = (df['gender'].str.strip().str.lower() == 'male').astype(int)

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Got: {list(df.columns)}")

print(f"Shape: {df.shape}  Prevalence: {df[TARGET_COL].mean():.3f}")
df.head(3)

In [ ]:
print(f"Missing:\n{df.isnull().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")

numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    n = ((df[col] < Q1 - 1.5*(Q3-Q1)) | (df[col] > Q3 + 1.5*(Q3-Q1))).sum()
    if n > 0:
        print(f"{col}: {n} outliers ({n/len(df)*100:.1f}%)")

## Feature selection

In [ ]:
ID_LIKE = {"id", "patient_id"}
feature_cols = [
    c for c in df.columns
    if c != TARGET_COL and c.lower() not in ID_LIKE and not c.lower().endswith("id")
]
print("Features:", feature_cols)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cols = feature_cols + [TARGET_COL]
corr = df[cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.3f')
plt.title('Feature Correlations')
plt.tight_layout()
plt.show()

target_corr = corr[TARGET_COL].abs().sort_values(ascending=False).drop(TARGET_COL)
print(f"Correlations with {TARGET_COL}:")
print(target_corr.head(8).to_string())

## Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df[feature_cols].copy()
y = df[TARGET_COL].astype(int).copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Prevalence train/test: {y_train.mean():.3f} / {y_test.mean():.3f}")

In [ ]:
import numpy as np
import pandas as pd

def add_clinical_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()

    g0  = X.get("glucose_fasting")
    g1  = X.get("ogtt_1h_glucose")
    g2  = X.get("ogtt_2h_glucose")
    bmi = X.get("bmi")
    ins = X.get("insulin_fasting")
    cpe = X.get("c_peptide_fasting")
    age = X.get("age")

    if g0 is not None and g1 is not None:
        X["ogtt_delta_1h"]   = (g1 - g0).astype(float)
        X["ogtt_slope_0_1h"] = (g1 - g0).astype(float)
    if g0 is not None and g2 is not None:
        X["ogtt_delta_2h"]   = (g2 - g0).astype(float)
        X["ogtt_slope_0_2h"] = (g2 - g0).astype(float)
    if g1 is not None and g2 is not None:
        X["ogtt_slope_1_2h"] = (g2 - g1).astype(float)
    if g0 is not None and g1 is not None and g2 is not None:
        X["ogtt_auc_trap"] = (0.5*(g0 + g1) + 0.5*(g1 + g2)).astype(float)

    if ins is not None and g0 is not None:
        X["homa_ir"] = (ins * g0 / 405.0).astype(float)

    if g0 is not None:
        X["fpg_prediabetes"] = ((g0 >= 100) & (g0 < 126)).astype(int)
        X["fpg_diabetes"]    = (g0 >= 126).astype(int)
    if g2 is not None:
        X["ogtt2h_prediabetes"] = ((g2 >= 140) & (g2 < 200)).astype(int)
        X["ogtt2h_diabetes"]    = (g2 >= 200).astype(int)
    if bmi is not None:
        X["bmi_overweight"] = ((bmi >= 25) & (bmi < 30)).astype(int)
        X["bmi_obese"]      = (bmi >= 30).astype(int)

    if ins is not None and cpe is not None:
        X["insulin_cpep_ratio"] = (ins / (cpe + 1e-6)).astype(float)
    if bmi is not None and g0 is not None:
        X["bmi_x_fpg"] = (bmi * g0).astype(float)
    if bmi is not None and age is not None:
        X["bmi_x_age"] = (bmi * age).astype(float)

    for col in ["age", "bmi", "glucose_fasting"]:
        if col in X.columns:
            X[f"{col}_sq"] = X[col].astype(float) ** 2

    for col in ["insulin_fasting", "c_peptide_fasting", "glucose_fasting"]:
        if col in X.columns:
            X[f"log1p_{col}"] = np.log1p(X[col].astype(float).clip(lower=0))

    return X

## Preprocessing

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_cols = X_train.select_dtypes(include="number").columns.tolist()
print("num_cols:", num_cols)

# scaled (LR, NN)
pre_linear = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
])

# no scaling (trees)
pre_tree = SimpleImputer(strategy="median")

# same as linear, kept separate so NN gets its own fitted scaler
pre_nn = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
])

## Evaluation

In [ ]:

import numpy as np
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             precision_recall_curve, precision_score, recall_score, confusion_matrix)
import matplotlib.pyplot as plt

def pr_opt_threshold(y_true, proba):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    if len(thr) <= 1:
        return 0.5
    J = prec + rec - 1
    return float(thr[np.argmax(J[:-1])])

def evaluate_model(model, X_te, y_te, name="Model"):
    proba = model.predict_proba(X_te)[:, 1]
    thr = pr_opt_threshold(y_te, proba)
    pred = (proba >= thr).astype(int)
    auroc = roc_auc_score(y_te, proba)
    ap    = average_precision_score(y_te, proba)
    brier = brier_score_loss(y_te, proba)
    prev  = float(np.mean(y_te))
    prec0 = precision_score(y_te, pred, zero_division=0)
    rec0  = recall_score(y_te, pred, zero_division=0)
    lift  = (prec0/prev) if prev>0 else float("nan")
    print(f"[{name}] AUROC={auroc:.4f}  AP={ap:.4f}  Prec={prec0:.3f}  Rec={rec0:.3f}  "
          f"Brier={brier:.4f}  Prev={prev:.3f}  Thr={thr:.3f}  Lift={lift:.2f}x")
    return dict(name=name, auroc=auroc, ap=ap, precision=prec0, recall=rec0,
                brier=brier, prev=prev, thr=thr, lift=lift)

def diag_plots(model, X, y, title="Model"):
    proba = model.predict_proba(X)[:,1]
    cm = confusion_matrix(y, (proba >= 0.5).astype(int))
    print("Confusion matrix @0.50:\n", cm)
    prec, rec, _ = precision_recall_curve(y, proba)
    plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"PR Curve - {title}")
    plt.grid(True); plt.show()


## Models

### Logistic Regression

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

lr_pipeline = Pipeline([
    ("pre", pre_linear),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs"))
])
lr_pipeline.fit(X_train, y_train)
metrics_lr = evaluate_model(lr_pipeline, X_test, y_test, name="LogisticRegression")
# diag_plots(lr_pipeline, X_test, y_test, title="LogisticRegression")


[LogisticRegression] AUROC=0.9582  AP=0.9472  Prec=0.872  Rec=0.846  Brier=0.0811  Prev=0.380  Thr=0.627  Lift=2.29x


### RandomForest

In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline([
    ("pre", pre_tree),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=1,              
        random_state=42,
    ))
])
rf_pipeline.fit(X_train, y_train)
metrics_rf = evaluate_model(rf_pipeline, X_test, y_test, name="RandomForest")
# diag_plots(rf_pipeline, X_test, y_test, title="RandomForest")


[RandomForest] AUROC=1.0000  AP=1.0000  Prec=1.000  Rec=0.999  Brier=0.0005  Prev=0.380  Thr=0.943  Lift=2.63x


### XGBoost

In [ ]:
try:
    import xgboost as xgb
    from sklearn.pipeline import Pipeline
    xgb_pipeline = Pipeline([
        ("pre", pre_tree),
        ("clf", xgb.XGBClassifier(
            tree_method="hist",
            n_estimators=800, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, random_state=42, eval_metric="auc", n_jobs=1
        ))
    ])
    xgb_pipeline.fit(X_train, y_train)
    metrics_xgb = evaluate_model(xgb_pipeline, X_test, y_test, name="XGBoost")
   
except Exception as e:
    print("XGBoost unavailable:", e)
    metrics_xgb = None

### Neural Net

In [ ]:

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    import numpy as np
    np.random.seed(42); tf.random.set_seed(42)

    # Fit preprocessor and transform
    Xtr = pre_nn.fit_transform(X_train)
    Xte = pre_nn.transform(X_test)
    n_in = Xtr.shape[1]

    def build_mlp(n_in, hidden=(256,128), dropout=0.15, lr=1e-3):
        inp = keras.Input(shape=(n_in,))
        x = inp
        for h in hidden:
            x = layers.Dense(h, activation="relu")(x)
            x = layers.Dropout(dropout)(x)
        out = layers.Dense(1, activation="sigmoid")(x)
        model = keras.Model(inp, out)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                      loss="binary_crossentropy",
                      metrics=[keras.metrics.AUC(name="auc"),
                               keras.metrics.AUC(name="auprc", curve="PR")])
        return model

    pos_rate = float(y_train.mean())
    class_weight = {0: 1.0, 1: (1.0 / max(pos_rate, 1e-6))}
    early_stop = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True)

    nn_model = build_mlp(n_in)
    nn_model.fit(Xtr, y_train.values, epochs=30, batch_size=1024, validation_split=0.1,
                 class_weight=class_weight, callbacks=[early_stop], verbose=0)

    class KerasAdapter:
        def __init__(self, model, pre): self.model, self.pre = model, pre
        def predict_proba(self, X):
            import numpy as np
            Xp = self.pre.transform(X)
            p = self.model.predict(Xp, batch_size=1024, verbose=0).ravel()
            return np.c_[1 - p, p]

    nn_adapter = KerasAdapter(nn_model, pre_nn)
    metrics_nn = evaluate_model(nn_adapter, X_test, y_test, name="NeuralNet (Keras)")
    # diag_plots(nn_adapter, X_test, y_test, title="NeuralNet (Keras)")
except Exception as e:
    print("TensorFlow unavailable or errored:", e)
    metrics_nn = None


[NeuralNet (Keras)] AUROC=1.0000  AP=0.9999  Prec=0.999  Rec=0.997  Brier=0.0019  Prev=0.380  Thr=0.792  Lift=2.63x


In [ ]:
def plot_feature_importance(model, feature_names, title="Feature Importance"):
    preprocessor = model.named_steps['pre']
    preprocessor.fit(X_train)

    feature_names_out = []
    for name, transformer, features in preprocessor.transformers_:
        if name != 'remainder':
            if hasattr(transformer, 'get_feature_names_out'):
                feature_names_out.extend(transformer.get_feature_names_out())
            else:
                feature_names_out.extend(features)

    importance = model.named_steps['clf'].feature_importances_
    indices = np.argsort(importance)[::-1]

    plt.figure(figsize=(12, 6))
    plt.title(title)
    plt.bar(range(len(importance)), importance[indices])
    plt.xticks(range(len(importance)), [feature_names_out[i] for i in indices], rotation=45)
    plt.tight_layout()
    plt.show()

plot_feature_importance(rf_pipeline, feature_cols, "Random Forest Feature Importance")

## Comparison

In [ ]:

import pandas as pd
rows = []
for m in ("metrics_lr", "metrics_rf", "metrics_xgb", "metrics_nn"):
    if m in globals() and globals()[m] is not None:
        rows.append(globals()[m])
cmp_df = pd.DataFrame(rows).set_index("name")
display(cmp_df.sort_values("auroc", ascending=False))


,auroc,ap,precision,recall,brier,prev,thr,lift
name,,,,,,,,
RandomForest,0.999975,0.999961,1.000000,0.998737,0.000482,0.380155,0.943101,2.630506
XGBoost,0.999968,0.999950,0.999947,0.995686,0.001356,0.380155,0.895628,2.630367
NeuralNet (Keras),0.999954,0.999929,0.998788,0.997028,0.001880,0.380155,0.792255,2.627317
LogisticRegression,0.958229,0.947194,0.871921,0.845602,0.081080,0.380155,0.627055,2.293595


## Save

In [ ]:

# import joblib, pandas as pd, numpy as np
# def save_model(obj, path):
#     joblib.dump(obj, path); print("Saved model ->", path)

# def save_preds(model, X, y, path="test_predictions.csv"):
#     p = model.predict_proba(X)[:,1]
#     pd.DataFrame({"y_true": y.values, "y_proba": p}).to_csv(path, index=False)
#     print("Saved predictions ->", path)

# # Examples (uncomment to use):
# # save_model(lr_pipeline, "model_logreg.joblib")
# # save_model(rf_pipeline, "model_rf.joblib")
# # if 'xgb_pipeline' in globals(): save_model(xgb_pipeline, "model_xgb.joblib")
# # if 'nn_adapter' in globals(): save_model(nn_adapter, "model_nn_adapter.joblib")
# # save_preds(lr_pipeline, X_test, y_test, "lr_test_preds.csv")


## Calibration

In [ ]:

from sklearn.calibration import CalibratedClassifierCV

cal_lr = CalibratedClassifierCV(lr_pipeline, method="isotonic", cv=3)
cal_lr.fit(X_train, y_train)
metrics_lr_cal = evaluate_model(cal_lr, X_test, y_test, name="LogReg+Isotonic")


[LogReg+Isotonic] AUROC=0.9582  AP=0.9467  Prec=0.870  Rec=0.847  Brier=0.0744  Prev=0.380  Thr=0.440  Lift=2.29x


In [ ]:
from sklearn.inspection import permutation_importance

def plot_permutation_importance(model, X_test, y_test, title="Permutation Importance"):
    perm = permutation_importance(model, X_test, y_test, scoring='roc_auc', n_repeats=5, random_state=42, n_jobs=1)
    indices = np.argsort(perm.importances_mean)[::-1]

    plt.figure(figsize=(10, 6))
    plt.title(title)
    plt.barh(range(len(indices)), perm.importances_mean[indices])
    plt.yticks(range(len(indices)), [X_test.columns[i] for i in indices])
    plt.xlabel('ROC AUC drop')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

plot_permutation_importance(lr_pipeline, X_test, y_test, "Logistic Regression - Permutation Importance")

## Drift check

In [ ]:

print("Train prevalence:", round(y_train.mean(),3), " | Test prevalence:", round(y_test.mean(),3))


Train prevalence: 0.38  | Test prevalence: 0.38


## Cross-validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "neg_brier": "neg_brier_score",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_summary(name, estimator, X, y):
    out = cross_validate(estimator, X, y, cv=cv, scoring=scoring, n_jobs=1, return_train_score=False)
    s  = {k.replace("test_", "") + "_mean": np.mean(v) for k, v in out.items() if k.startswith("test_")}
    sd = {k.replace("test_", "") + "_std":  np.std(v)  for k, v in out.items() if k.startswith("test_")}
    return {"name": name, **s, **sd}

rows = [cv_summary("LogisticRegression", lr_pipeline, X, y),
        cv_summary("RandomForest", rf_pipeline, X, y)]
try:
    rows.append(cv_summary("XGBoost", xgb_pipeline, X, y))
except Exception:
    pass

cv_df = pd.DataFrame(rows).set_index("name")
display(cv_df)

In [ ]:
from sklearn.model_selection import GridSearchCV

def quick_rf_tuning(X_train, y_train):
    param_grid = {
        'clf__n_estimators': [200, 400],
        'clf__max_depth': [10, None],
        'clf__min_samples_leaf': [1, 2]
    }
    pipe = Pipeline([
        ("pre", pre_tree),
        ("clf", RandomForestClassifier(class_weight="balanced_subsample", n_jobs=1, random_state=42))
    ])
    gs = GridSearchCV(pipe, param_grid, cv=3, scoring='roc_auc', n_jobs=1, verbose=1)
    gs.fit(X_train, y_train)
    print(f"Best params: {gs.best_params_}  CV score: {gs.best_score_:.4f}")
    return gs.best_estimator_

best_rf = quick_rf_tuning(X_train, y_train)
metrics_rf_tuned = evaluate_model(best_rf, X_test, y_test, name="RandomForest-Tuned")

## Train/val/test split

In [ ]:
try:
    _ = (X_train, X_val, X_test, y_train, y_val, y_test)
    print("Using existing splits.")
except NameError:
    from sklearn.model_selection import train_test_split

    TARGET_COL = "diabetes_label" if "diabetes_label" in df.columns else "target"
    X_full = df.drop(columns=[TARGET_COL]).copy()
    y_full = df[TARGET_COL].astype(int).copy()

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X_full, y_full, test_size=0.2, stratify=y_full, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.2, stratify=y_train_val, random_state=42
    )

    print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

    num_cols = X_train.select_dtypes(include="number").columns.tolist()
    pre_linear = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
    pre_tree = SimpleImputer(strategy="median")
    pre_nn = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])

## EDA plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(5,4))
sns.countplot(x=y_full, hue=y_full, palette="viridis", legend=False)
plt.title("Target Distribution")
plt.xlabel("Diabetes Label"); plt.ylabel("Count")
plt.show()

num_df = df.select_dtypes(include=['number'])
plt.figure(figsize=(10,8))
sns.heatmap(num_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

if "bmi" in df.columns:
    plt.figure(figsize=(7,5))
    sns.histplot(data=df, x="bmi", hue="diabetes_label", multiple="stack", palette="viridis")
    plt.title("BMI by Diabetes Label"); plt.xlabel("BMI"); plt.ylabel("Count")
    plt.show()

## Pairplot

In [ ]:

import seaborn as sns
import pandas as pd

# Select up to 6 numeric features + target for a readable pairplot
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != "diabetes_label"]
subset = ["age","bmi","glucose_fasting","insulin_fasting","ogtt_2h_glucose","c_peptide_fasting"]
subset = [c for c in subset if c in num_cols][:6]
if "diabetes_label" in df.columns:
    plot_df = df[subset + ["diabetes_label"]].copy()
    sns.pairplot(plot_df, hue="diabetes_label", corner=True, diag_kind="hist")
else:
    plot_df = df[subset].copy()
    sns.pairplot(plot_df, corner=True, diag_kind="hist")
plt.show()


## Threshold tuning

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_curve, auc

def pr_opt_threshold(y_true, proba):
    p, r, t = precision_recall_curve(y_true, proba)
    if len(t) <= 1:
        return 0.5, p, r, t
    J = p + r - 1
    return float(t[np.argmax(J[:-1])]), p, r, t

def plot_pr_roc(y_true, proba, title="Model on Validation"):
    p, r, _ = precision_recall_curve(y_true, proba)
    fpr, tpr, _ = roc_curve(y_true, proba)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(r, p); ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision")
    ax[0].set_title(f"PR curve (AP={auc(r,p):.3f})"); ax[0].grid(True)
    ax[1].plot(fpr, tpr); ax[1].plot([0,1],[0,1],'--')
    ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR")
    ax[1].set_title(f"ROC curve (AUC={auc(fpr,tpr):.3f})"); ax[1].grid(True)
    plt.suptitle(title); plt.show()

chosen = next((globals()[n] for n in ["xgb_pipeline", "rf_pipeline", "lr_pipeline"] if n in globals()), None)

if chosen is None:
    print("No trained model found.")
else:
    proba_val = chosen.predict_proba(X_val)[:,1]
    thr_opt, p, r, t = pr_opt_threshold(y_val, proba_val)
    print(f"Optimal threshold: {thr_opt:.3f}")
    plot_pr_roc(y_val, proba_val, title=f"{type(chosen.named_steps.get('clf', chosen)).__name__} on Validation")

## Feature importances

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def get_feature_names(ct):
    try:
        return list(ct.get_feature_names_out())
    except Exception:
        names = []
        for name, trans, cols in ct.transformers_:
            if name == "remainder":
                continue
            if hasattr(trans, "get_feature_names_out"):
                try:
                    names.extend(list(trans.get_feature_names_out(cols)))
                except Exception:
                    names.extend(cols if isinstance(cols, list) else [cols])
            else:
                names.extend(cols if isinstance(cols, list) else [cols])
        return names

def plot_top_series(s, title, k=20):
    s = s.sort_values(ascending=False).head(k)
    plt.figure(figsize=(8, max(3, 0.35*len(s))))
    s[::-1].plot(kind="barh")
    plt.title(title); plt.tight_layout(); plt.show()

plotted = False

if "lr_pipeline" in globals():
    pre = lr_pipeline.named_steps.get("pre")
    clf = lr_pipeline.named_steps.get("clf")
    if pre is not None and clf is not None and hasattr(clf, "coef_"):
        coefs = pd.Series(np.abs(clf.coef_.ravel()), index=get_feature_names(pre))
        plot_top_series(coefs, "Logistic Regression - |coef|")
        plotted = True

if "rf_pipeline" in globals():
    pre = rf_pipeline.named_steps.get("pre")
    clf = rf_pipeline.named_steps.get("clf")
    if pre is not None and clf is not None and hasattr(clf, "feature_importances_"):
        imps = pd.Series(clf.feature_importances_, index=get_feature_names(pre))
        plot_top_series(imps, "RandomForest - Feature Importance")
        plotted = True

if "xgb_pipeline" in globals():
    pre = xgb_pipeline.named_steps.get("pre")
    clf = xgb_pipeline.named_steps.get("clf")
    if pre is not None and clf is not None and hasattr(clf, "feature_importances_"):
        imps = pd.Series(clf.feature_importances_, index=get_feature_names(pre))
        plot_top_series(imps, "XGBoost - Feature Importance")
        plotted = True

if not plotted:
    print("Train a model first.")

## Learning curve

In [ ]:

from sklearn.model_selection import learning_curve, StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt

if "lr_pipeline" not in globals():
    print("Train lr_pipeline first to run learning curve.")
else:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_sizes, train_scores, test_scores = learning_curve(
        lr_pipeline, X_train, y_train, cv=cv,
        scoring="roc_auc", n_jobs=1,
        train_sizes=np.linspace(0.1, 1.0, 6), shuffle=True, random_state=42
    )
    train_mean = train_scores.mean(axis=1)
    test_mean = test_scores.mean(axis=1)

    plt.figure(figsize=(7,5))
    plt.plot(train_sizes, train_mean, marker="o", label="Train ROC AUC")
    plt.plot(train_sizes, test_mean, marker="o", label="CV ROC AUC")
    plt.xlabel("Training examples"); plt.ylabel("ROC AUC")
    plt.title("Learning Curve - Logistic Regression")
    plt.legend(); plt.grid(True); plt.show()


## Validation curve

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
import matplotlib.pyplot as plt

if "rf_pipeline" not in globals():
    print("Train rf_pipeline first.")
else:
    depths = [None, 4, 6, 8, 10, 12, 16]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for d in depths:
        est = Pipeline([
            ("pre", rf_pipeline.named_steps["pre"]),
            ("clf", RandomForestClassifier(n_estimators=400, max_depth=d, min_samples_leaf=2,
                                           class_weight="balanced_subsample", n_jobs=1, random_state=42))
        ])
        scores.append(cross_val_score(est, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1).mean())

    plt.figure(figsize=(7,5))
    x = list(range(len(depths)))
    plt.plot(x, scores, marker="o")
    plt.xticks(x, [str(d) for d in depths])
    plt.xlabel("max_depth"); plt.ylabel("CV ROC AUC")
    plt.title("Validation Curve - RandomForest max_depth")
    plt.grid(True); plt.show()

In [ ]:
import joblib, numpy as np, pandas as pd
from sklearn.metrics import precision_recall_curve

def _proba1d(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        return np.asarray(p)[:, 1].ravel()
    if hasattr(model, "decision_function"):
        s = np.asarray(model.decision_function(X)).ravel()
        return 1.0 / (1.0 + np.exp(-s))
    raise AttributeError("Model has neither predict_proba nor decision_function.")

def _pick_threshold_from_val(model, X_val=None, y_val=None, default=0.50):
    if X_val is None or y_val is None:
        print(f"No validation split detected - using default threshold={default:.2f}")
        return float(default)
    proba = _proba1d(model, X_val)
    prec, rec, thr = precision_recall_curve(y_val, proba)
    if thr.size == 0:
        print(f"Could not compute PR thresholds - using default threshold={default:.2f}")
        return float(default)
    J = prec[:-1] + rec[:-1] - 1.0
    t = float(thr[int(np.nanargmax(J))])
    print(f"Threshold from validation (PR-opt): {t:.3f}")
    return t

def save_model(model, path="model.joblib"):
    joblib.dump(model, path)
    print(f"Saved model -> {path}")

def save_test_predictions(model, X_test, y_test, path="test_predictions.csv", threshold=None):
    proba = _proba1d(model, X_test)
    out = {"y_true": pd.Series(y_test).astype(int).values, "y_proba": proba}
    if threshold is not None:
        out["y_pred"] = (proba >= float(threshold)).astype(int)
        print(f"Used threshold = {float(threshold):.3f} for y_pred")
    pd.DataFrame(out).to_csv(path, index=False)
    print(f"Saved predictions -> {path} (rows={len(y_test)})")

chosen = lr_pipeline
# chosen = rf_pipeline
# chosen = xgb_pipeline

X_val_local = globals().get("X_val")
y_val_local = globals().get("y_val")
thr = _pick_threshold_from_val(chosen, X_val_local, y_val_local, default=0.50)

save_model(chosen, "model.joblib")
save_test_predictions(chosen, X_test, y_test, "test_predictions.csv", threshold=thr)

In [ ]:
import sys, subprocess, numpy as np, pandas as pd, joblib
from IPython.display import display, HTML
from sklearn.metrics import precision_recall_curve

try:
    import ipywidgets as widgets
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets", "--quiet"])
    import ipywidgets as widgets

def _proba1d(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        return np.asarray(p)[:, 1].ravel()
    if hasattr(model, "decision_function"):
        s = np.asarray(model.decision_function(X)).ravel()
        return 1.0 / (1.0 + np.exp(-s))
    raise AttributeError("Model has neither predict_proba nor decision_function.")

def _thr_from_val_or_default(model, default=0.50):
    X_val = globals().get("X_val")
    y_val = globals().get("y_val")
    if X_val is None or y_val is None:
        return float(default), "default"
    if "add_clinical_features" in globals() and callable(add_clinical_features) and "ogtt_delta_1h" not in X_val.columns:
        Xv = add_clinical_features(X_val)
    else:
        Xv = X_val
    proba_v = _proba1d(model, Xv)
    prec, rec, thr = precision_recall_curve(y_val, proba_v)
    if thr.size == 0:
        return float(default), "default"
    J = prec[:-1] + rec[:-1] - 1.0
    return float(thr[int(np.nanargmax(J))]), "validation"

w_model_path = widgets.Text(value="model.joblib", description="Model path:", layout=widgets.Layout(width="350px"))

w_age   = widgets.BoundedIntText(value=52, min=1, max=120, step=1, description="Age (y):")
w_gender= widgets.Dropdown(options=["Male", "Female"], value="Male", description="Gender:")
w_bmi   = widgets.BoundedFloatText(value=29.4, min=10, max=80, step=0.1, description="BMI (kg/m²):")

w_fpg   = widgets.BoundedFloatText(value=118, min=40, max=500, step=1.0, description="FPG (mg/dL):")
w_ins   = widgets.BoundedFloatText(value=12.3, min=0, max=500, step=0.1, description="Insulin (µU/mL):")
w_cpep  = widgets.BoundedFloatText(value=1.8, min=0, max=20, step=0.1, description="C-pep (ng/mL):")

w_ogtt1 = widgets.BoundedFloatText(value=165, min=40, max=500, step=1.0, description="OGTT 1h (mg/dL):")
w_ogtt2 = widgets.BoundedFloatText(value=185, min=40, max=500, step=1.0, description="OGTT 2h (mg/dL):")

w_auto_thr = widgets.Checkbox(value=True, description="Auto threshold from validation (if available)")
w_thr      = widgets.FloatSlider(value=float(globals().get("thr", 0.50)),
                                 min=0.0, max=1.0, step=0.01, readout=True,
                                 description="Manual thr:")

w_predict  = widgets.Button(description="Predict", button_style="primary", icon="check")
w_out      = widgets.Output()

def _on_auto_thr_change(change):
    w_thr.disabled = change["new"]
_on_auto_thr_change({"new": w_auto_thr.value})
w_auto_thr.observe(_on_auto_thr_change, names="value")

def _predict_clicked(_):
    w_out.clear_output()
    with w_out:
        try:
            model = joblib.load(w_model_path.value)
        except Exception as e:
            print("Could not load model:", e)
            return

        raw = {
            "age": w_age.value,
            "gender": w_gender.value,
            "bmi": w_bmi.value,
            "glucose_fasting": w_fpg.value,
            "insulin_fasting": w_ins.value,
            "c_peptide_fasting": w_cpep.value,
            "ogtt_1h_glucose": w_ogtt1.value,
            "ogtt_2h_glucose": w_ogtt2.value,
        }
        X_new = pd.DataFrame([raw])
        X_new['gender'] = (X_new['gender'].str.strip().str.lower() == 'male').astype(int)

        try:
            X_new_fe = add_clinical_features(X_new)
        except NameError:
            print("add_clinical_features not defined - using raw inputs. Results may not match training.")
            X_new_fe = X_new

        if w_auto_thr.value:
            thr, source = _thr_from_val_or_default(model, default=float(globals().get("thr", 0.50)))
        else:
            thr, source = float(w_thr.value), "manual"

        try:
            proba = float(_proba1d(model, X_new_fe)[0])
        except Exception as e:
            print("Prediction failed:", e)
            return
        pred = int(proba >= thr)

        risk = "AT RISK" if pred == 1 else "SAFE"
        color = "#c0392b" if pred == 1 else "#27ae60"
        display(HTML(f"""
        <div style="border:1px solid {color}; padding:14px; border-radius:10px; max-width:560px;">
            <b>Patient readings</b>
            <div style="font-size:16px; line-height:1.6; margin-top:8px;">
                <b>Status:</b> <span style="color:{color}; font-weight:700;">{risk}</span><br/>
                <b>Probability:</b> {proba:.3f}<br/>
                <b>Threshold:</b> {thr:.2f} <small>({source})</small><br/>
            </div>
        </div>
        """))

w_predict.on_click(_predict_clicked)

row1 = widgets.HBox([w_age, w_gender, w_bmi])
row2 = widgets.HBox([w_fpg, w_ins, w_cpep])
row3 = widgets.HBox([w_ogtt1, w_ogtt2])
row4 = widgets.HBox([w_auto_thr, w_thr])
row5 = widgets.HBox([w_model_path, w_predict])

box = widgets.VBox([
    widgets.HTML("<b>Patient readings</b>"),
    row1, row2, row3,
    widgets.HTML("<b>Threshold:</b>"),
    row4,
    row5,
    w_out
])

display(box)